# Audit an attribution graph on your own prompt

Attribution graphs explain a model's prediction with arrows: "this feature pushes the answer up by this much". This notebook checks those claims on GPT-2 small. For the strongest arrows it switches the feature off and compares three answers:

- **graph**: what today's attribution graph predicts;
- **fixed**: the same graph with the two missing derivatives (LayerNorm and attention) put back;
- **GPT-2**: what the real model does.

Runs on the free Colab CPU. The first run downloads GPT-2 small and its transcoders (about 2.3 GB, a few minutes).

[Live demo](https://spiliosdimakopoulos.github.io/twinlens/) · [Report](https://spiliosdimakopoulos.github.io/twinlens/report/) · [Code](https://github.com/SpiliosDimakopoulos/twinlens)

In [ ]:
%pip install -q "twinlens @ git+https://github.com/SpiliosDimakopoulos/twinlens"

In [ ]:
from twinlens.cli import audit, _print

PROMPT = "When Mary and John went to the store, John gave a drink to"  # change me
report = audit(PROMPT, top=8, edit="ablate")
_print(report)

**Reading the table.** Each row is one arrow of the graph. The three numbers are the change in log-probability of GPT-2's answer when that feature is switched off (−1 ≈ the answer becomes 2.7× less likely). A verdict of `WRONG SIGN` means the graph predicts the opposite of what GPT-2 does. `trust` is the graph's estimated chance of being close, computed before GPT-2 is run.

The same audit is available from a terminal: `twinlens audit "your prompt" --top 12`.

In [ ]:
import pandas as pd
pd.DataFrame(report["edges"])[["layer", "token", "feature", "graph", "fixed", "gpt2", "graph_verdict", "trust"]].round(3)